In [1]:
import random
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision

sns.set(style="whitegrid")

## Reproducibility

In [2]:

SEED = 2019
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark = False

## (Down-)Loading the dataset

In [3]:

transform = torchvision.transforms.transforms.Compose([
    torchvision.transforms.transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True, transform=transform)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
trainset, testset

Files already downloaded and verified
Files already downloaded and verified


(Dataset CIFAR10
     Number of datapoints: 50000
     Root location: datasets
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
            ), Dataset CIFAR10
     Number of datapoints: 10000
     Root location: datasets
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
            ))

## Setting up

In [4]:

BATCH_SIZE = 50
N_BATCHES_IN_TRAIN_SET = len(trainset) // BATCH_SIZE
N_BATCHES_IN_TEST_SET = len(testset) // BATCH_SIZE

NUM_WORKERS = 8

# Fixed learning rate
LR = 0.005

# CIFAR10 number of classes
NUM_CLASS = 10

# CIFAR10 image metadata
CHANNEL, IMAGE_SIZE, _ = trainset[0][0].shape
print("images are:", IMAGE_SIZE, CHANNEL)

TRAIN_EPOCHS = 5

images are: 32 3


In [5]:
PATH_TO_EFFICIENT_NET = "/home/landelle/EfficientNet-PyTorch"
import sys
sys.path.append(PATH_TO_EFFICIENT_NET)

In [6]:

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## Loading a pretrained EfficientNet model

In [7]:
from efficientnet_pytorch import EfficientNet
model = EfficientNet.from_pretrained('efficientnet-b0')

Loaded pretrained weights for efficientnet-b0


In [8]:
import main

In [16]:

def train(device, model, optimizer, criterion, lam, n_epochs=None, n_batches=None):
    # switch to train mode
    model.train()
    n_epochs = n_epochs if n_epochs else TRAIN_EPOCHS
    for epoch in range(1, n_epochs + 1):
        print("Epoch[{}/{}]".format(epoch, n_epochs))
        for batch_id, (images, labels) in enumerate(trainloader):
            if n_batches and batch_id > n_batches:
                break
            labels, images = labels.to(device), images.to(device)
            images, labels_a, labels_b, lam = main.mixup_data(images, labels, lam)
            outputs = model(images)
            loss = main.mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_id % 50 == 0:
                print('Loss :{:.4f} Epoch[{}/{}] Batch[{}/{}] batch_shape:{}'.format(
                    loss.item(), epoch, n_epochs, batch_id, N_BATCHES_IN_TRAIN_SET, images.shape))

In [17]:

def test(device, model):
    # switch to evaluate mode
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        # Output test accuracy
        test_acc = 100 * correct / total
        print('Test Accuracy of the model on the test images: {} %'.format(test_acc))
        return test_acc

In [18]:

def collect_results(n_epochs=10, n_batches=None, half=False):
    """
    n_batches: None trains on whole dataset otherwise it trains on n_batches of BATCH_SIZE per epoch    
    
    """
    USE_CUDA = True

    test_accs = {}
    for lambda_ in [x*.1 for x in range(11)]:
        if half and x>.5:
            break
        print("Trying lambda=", lambda_)

        device = torch.device('cuda' if USE_CUDA else 'cpu')
        #model = models.My_Model(NUM_CLASS).to(device)
        model = EfficientNet.from_pretrained('efficientnet-b0').to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = torch.nn.CrossEntropyLoss()

        train(device, model, optimizer, criterion, lambda_, n_epochs, n_batches)
        test_accs[lambda_] = test(device, model)

        del device, model, optimizer, criterion
    test_accs_dict = {str(k)[:3]:[v] for k, v in test_accs.items()}
    test_accs_df = pd.DataFrame.from_dict(test_accs_dict).transpose()
    test_accs_df.reset_index(inplace=True)
    test_accs_df.rename(columns={0:'Test accuracy', 'index':'Lambda'}, inplace=True)
    return test_accs_df
    

## Visualizing Results

In [ ]:
test_accs2 = collect_results(n_epochs=1, n_batches=50)

Trying lambda= 0.0
Loaded pretrained weights for efficientnet-b0
Epoch[1/1]
Loss :10.9591 Epoch[1/1] Batch[0/1000] batch_shape:torch.Size([50, 3, 32, 32])
Loss :2.3713 Epoch[1/1] Batch[50/1000] batch_shape:torch.Size([50, 3, 32, 32])
Test Accuracy of the model on the test images: 10.0 %
Trying lambda= 0.1
Loaded pretrained weights for efficientnet-b0
Epoch[1/1]
Loss :10.8951 Epoch[1/1] Batch[0/1000] batch_shape:torch.Size([50, 3, 32, 32])
Loss :2.1005 Epoch[1/1] Batch[50/1000] batch_shape:torch.Size([50, 3, 32, 32])
Test Accuracy of the model on the test images: 9.83 %
Trying lambda= 0.2
Loaded pretrained weights for efficientnet-b0
Epoch[1/1]
Loss :11.1940 Epoch[1/1] Batch[0/1000] batch_shape:torch.Size([50, 3, 32, 32])


In [14]:
test_accs2

NameError: name 'test_accs2' is not defined

In [12]:
sns.relplot(x="Lambda", y="Test accuracy", hue="Test accuracy", data=test_accs2)

NameError: name 'test_accs2' is not defined